# The Beatles, Part A: Clean Data

**Beatles notebooks:** **A. Clean Data** · [B. Tidy Data](B_Beatles_Tidy_Data.ipynb) · [C. Find, Filter, Group](C_Beatles_Find_Filter_Group.ipynb) · [D. Charts and Graphs](D_Beatles_Charts_Graphs.ipynb)

**The big picture.** Real data is almost never ready to use. Before we can ask whether Harrison's songs sound different from Lennon's, or whether the hits of 1964 are still popular today, the computer has to read our data the way we would: "Live At The BBC" and "Live at the BBC" should count as the same album, a year should be a number rather than text, and an empty cell should be understood for what it actually means. Every one of these fixes is a **decision**, and each one shapes what our data can later tell us. That is why cleaning is part of the research, not a chore before it.

**Our data.** `Beatles_Belgrade_Complete.csv` combines **Spotify** audio measurements with metadata compiled at the **University of Belgrade** (songwriters, singers, albums, genre, style, theme and mood labels, and chart histories). Section 1 introduces it in detail.

**In this notebook you will learn to:**
* inspect a new dataset: its size, data types, and missing values
* decide what missing values *mean*, and then fill, drop, or keep them
* fix inconsistent values with three tools: `replace()`, `map()`, and `apply()`
* turn text labels into Python **lists**, ready for tidying
* correct data types and check for duplicates
* save the result as a Pickle for the next notebooks

**Input → Output:** `Beatles_Belgrade_Complete.csv` → **`beatles_clean.pkl`** (one row per song), used in Notebooks B, C, and D.

**Keep in mind:** as you work, note each cleaning decision and *why* you made it. Someone else should be able to follow, and question, your choices.

**Contents of this Notebook**

- **1.** Meet the Beatles Data
- **2.** Tidy Column Names
- **3.** Missing Data: Find, Decide, Fill or Drop
- **4.** Wrong or Inconsistent Values: `replace()`, `map()`, and `apply()`
- **5.** Simplifying a Column for Research: The Principal Songwriter
- **6.** Wrong Data Types
- **7.** Duplicates
- **8.** Save the Clean Data as a Pickle

Resources: [Pandas Docs](https://pandas.pydata.org/docs/) · [W3Schools Pandas](https://www.w3schools.com/python/pandas/default.asp) · [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)

## 1. Meet the Beatles Data

Our file, `Beatles_Belgrade_Complete.csv`, combines two sources that you may have met separately before:

* **Spotify** audio features for each track: `Popularity`, `Duration`, `Key`, `Mode`, `Tempo`, and the 0–1 ratings `Valence`, `Danceability`, `Energy`, `Acousticness`, `Instrumentalness`, `Liveness`, `Speechiness`.
* **University of Belgrade** metadata: year, album debut, songwriter(s), lead vocal(s), whether the song is a cover, plus
  * **categorical labels** for `Genre`, `Styles`, `Themes`, and `Moods` (several labels per song!)
  * **chart and ranking data**: Billboard, UK charts (via Wikipedia and *The Guardian*), and several "Top 50 Beatles songs" lists (Rolling Stone, NME, *USA Today*, Vulture, and others).

Because the merging has already been done, we can go straight to cleaning.

### Research questions to keep in mind

Cleaning decisions always depend on what we want to ask. Across these four notebooks we will pursue questions like:

* How did the Beatles' **sound change over time** (energy, acousticness, valence by year or album)?
* Do **Lennon, McCartney, and Harrison** songs differ in genre, mood, or audio features?
* Does historical **chart success** (Billboard) relate to present-day **Spotify popularity**?
* Which **genres, styles, themes, and moods** are most typical of the band, and how do they cluster?
* How do **covers** differ from **original** Beatles songs?

In [1]:
import pandas as pd

# show all columns when we display a DataFrame
pd.set_option('display.max_columns', None)

### Load the CSV

The CSV sits in the same folder as this notebook, so we can load it with a simple relative path.

In [2]:
beatles_raw = pd.read_csv('Beatles_Belgrade_Complete.csv')
beatles_raw.shape

(285, 45)

In [3]:
beatles_raw.head(3)

,URI,Title,Year,Album,Popularity,Duration,Key,Mode,Tempo,Time_signature,Valence,Danceability,Energy,Loudness,Acousticness,Instrumentalness,Liveness,Speechiness,Album debut,Other releases,Single A side,Single B side,Single certification,Genre,Styles,Themes,Moods,Songwriter(s),Lead vocal(s),Cover,Covered by,Chart position UK (Wikipedia),Chart position US (Wikipedia),Highest position (The Guardian),Weeks on chart in UK (The Guardian),Weeks at No1 in UK (The Guardian),Highest position (Billboard),Weeks at No1 (Billboard),Top 50 (Billboard),Top 50 (Ultimate classic rock),Top 50 (Rolling Stone),Top 50 (NME),Top 50 (Top50songs.org),"Top 50 (USA today, 2017)","Top 50 (Vulture, by Bill Wyman)"
0,spotify:track:2FDEHIMkjxFLzj688M2I3h,(You're So Square) Baby I Don't Care - Studio Jam,NaN,The Beatles,29.0,43.0,9.0,1.0,112.173,4.0,0.353,0.411,0.840,-9.919,0.09910,0.000,0.364,0.0896,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,spotify:track:2HvTGx5fzFGpHSyRNvXd9T,12-bar Original,1965,Anthology 2,31.0,175.0,9.0,1.0,122.678,4.0,0.566,0.600,0.535,-8.178,0.00154,0.102,0.123,0.0298,Anthology 2,NaN,NaN,NaN,NaN,Blues,NaN,NaN,Passionate,"Lennon, McCartney, Harrison and Starkey",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,spotify:track:34dsKRJHIadrrNdCDtMwGn,A Beginning - Anthology 3 Version,NaN,Anthology 3,29.0,50.0,0.0,1.0,90.588,3.0,0.033,0.264,0.293,-16.498,0.95800,0.942,0.338,0.0377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


`info()` shows each column's **data type** and the number of **non-null** (not missing) values. Already we can see problems: `Year` is stored as text (`object`/`str`), not a number, and many columns have far fewer than 285 values.

In [4]:
beatles_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285 entries, 0 to 284
Data columns (total 45 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   URI                                  285 non-null    object 
 1   Title                                285 non-null    object 
 2   Year                                 277 non-null    object 
 3   Album                                284 non-null    object 
 4   Popularity                           285 non-null    float64
 5   Duration                             285 non-null    float64
 6   Key                                  285 non-null    float64
 7   Mode                                 285 non-null    float64
 8   Tempo                                285 non-null    float64
 9   Time_signature                       285 non-null    float64
 10  Valence                              285 non-null    float64
 11  Danceability                    

## 2. Tidy Column Names

Column names like `Songwriter(s)` or `Top 50 (USA today, 2017)` are awkward to type, and spaces and parentheses cause trouble in some Pandas methods. A common convention is **snake_case**: all lowercase, words joined by underscores.

Rather than renaming 45 columns by hand, we write a small **function** and apply it to every column name. This is the same approach we will use for cleaning values below.

In [5]:
def clean_column_name(name):
    name = name.lower()
    name = name.replace('(s)', '')           # 'songwriter(s)' -> 'songwriter'
    for character in ['(', ')', ',', '.']:   # remove punctuation
        name = name.replace(character, '')
    name = name.strip().replace(' ', '_')    # spaces -> underscores
    return name

# always work on a copy, so the raw data stays untouched
beatles = beatles_raw.copy()
beatles.columns = [clean_column_name(col) for col in beatles.columns]
list(beatles.columns)

['uri',
 'title',
 'year',
 'album',
 'popularity',
 'duration',
 'key',
 'mode',
 'tempo',
 'time_signature',
 'valence',
 'danceability',
 'energy',
 'loudness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'speechiness',
 'album_debut',
 'other_releases',
 'single_a_side',
 'single_b_side',
 'single_certification',
 'genre',
 'styles',
 'themes',
 'moods',
 'songwriter',
 'lead_vocal',
 'cover',
 'covered_by',
 'chart_position_uk_wikipedia',
 'chart_position_us_wikipedia',
 'highest_position_the_guardian',
 'weeks_on_chart_in_uk_the_guardian',
 'weeks_at_no1_in_uk_the_guardian',
 'highest_position_billboard',
 'weeks_at_no1_billboard',
 'top_50_billboard',
 'top_50_ultimate_classic_rock',
 'top_50_rolling_stone',
 'top_50_nme',
 'top_50_top50songsorg',
 'top_50_usa_today_2017',
 'top_50_vulture_by_bill_wyman']

## 3. Missing Data: Find, Decide, Fill or Drop

Missing data (`NaN`, `None`, `pd.NA`) is one of the most common problems in real datasets. String methods and many calculations will fail or give odd results when they meet a missing value, so we must **find** missing values and then **decide** what they mean before we **fill** or **drop** them.

### Find: how much is missing, and where?

In [6]:
missing = pd.DataFrame({
    'missing': beatles.isna().sum(),
    'percent': (beatles.isna().mean() * 100).round(1)
})
missing[missing['missing'] > 0].sort_values('missing', ascending=False)

,missing,percent
weeks_at_no1_billboard,270,94.7
weeks_at_no1_in_uk_the_guardian,269,94.4
highest_position_the_guardian,254,89.1
weeks_on_chart_in_uk_the_guardian,254,89.1
top_50_vulture_by_bill_wyman,243,85.3
chart_position_uk_wikipedia,240,84.2
top_50_nme,236,82.8
top_50_rolling_stone,236,82.8
top_50_billboard,236,82.8
highest_position_billboard,236,82.8


### Decide: not all missing data means the same thing

Reading down that table, we can see **three very different kinds** of missing data:

1. **Missing because the event never happened.** The chart and ranking columns (`top_50_billboard`, `chart_position_us_wikipedia`, `weeks_at_no1_billboard`, ...) are empty for most songs because most songs were never hits. `NaN` here is *information*: "did not chart." We should **not** fill these with `0`: a chart position of 0 would be meaningless (or would look like "better than No. 1"!). We will leave them as `NaN` and create a simple True/False column instead.
2. **Missing optional labels.** `themes`, `styles`, `moods`, `lead_vocal`, `album_debut`: some songs simply were not labelled. We can fill these with a sensible default.
3. **Whole rows with no Belgrade metadata.** Some Spotify tracks (studio jams, alternate takes) have no genre, songwriter, or year at all.

Let's look at the third group first. Here is a **filter** (more on these in Notebook C) showing rows with no `genre`:

In [7]:
beatles[beatles['genre'].isna()][['title', 'year', 'album', 'songwriter', 'genre', 'moods']]

,title,year,album,songwriter,genre,moods
0,(You're So Square) Baby I Don't Care - Studio Jam,NaN,The Beatles,NaN,NaN,NaN
2,A Beginning - Anthology 3 Version,NaN,Anthology 3,NaN,NaN,NaN
8,Across the Universe - Take 2 / Anthology 2 Ver...,NaN,Anthology 2,NaN,NaN,NaN
30,Besame Mucho,NaN,Anthology 1,NaN,NaN,NaN
35,Can You Take Me Back? - Take 1,NaN,The Beatles,NaN,NaN,NaN
83,"Hallelujah, I Love Her So",1960,Anthology 1,NaN,NaN,NaN
152,Los Paranoias - Studio Jam,NaN,The Beatles,NaN,NaN,NaN


### Drop: rows we cannot use

These seven tracks are outtakes and jams that exist only in the Spotify data. With no genre, songwriter, or year they cannot take part in any of our research questions, so we **drop** them. `your_df.dropna(subset=['your_column'])` drops rows that are missing values *in the listed columns only*.

> Dropping data is a research decision. Write down *why* you dropped it (as we do here), so others can evaluate your choice.

In [8]:
beatles = beatles.dropna(subset=['genre'])
beatles.shape

(278, 45)

### Check again: what is still missing in the descriptive columns?

In [9]:
descriptive_cols = ['year', 'album', 'album_debut', 'songwriter', 'lead_vocal',
                    'genre', 'styles', 'themes', 'moods']
beatles[descriptive_cols].isna().sum()

year            2
album           1
album_debut     3
songwriter      0
lead_vocal      5
genre           0
styles         21
themes         94
moods          18
dtype: int64

Let's look at the songs with no `lead_vocal`. The high Spotify `instrumentalness` scalar for several of them seems to suggest that these might be instrumentals.  We should review with our own ears to be sure, but we could replace the NaN values `lead_vocal` with something like `None (instrumental)`.

In [10]:
beatles[beatles['lead_vocal'].isna()][['title', 'year', 'songwriter', 'lead_vocal', 'instrumentalness']]

,title,year,songwriter,lead_vocal,instrumentalness
1,12-bar Original,1965,"Lennon, McCartney, Harrison and Starkey",NaN,0.102000
39,Cayenne,1960,McCartney,NaN,0.895000
46,Cry for a Shadow,1961,Lennon and Harrison,NaN,0.000154
67,Flying,1967,"Lennon, McCartney, Harrison and Starkey",NaN,0.880000
197,Revolution 9,1968,"Lennon, with Ono and Harrison",NaN,0.118000




### Fill: one column at a time, with values that makes sense

`fillna()` replaces missing values. Because each column has a different meaning, we fill them one at a time:

* `lead_vocal` → `'None (instrumental)'`
* `album` and `album_debut` → `'Unreleased'` (these are demos and songs the Beatles gave to other artists)
* `styles`, `themes`, `moods` → an **empty string** `''`. This keeps every value as text, so the cleaning steps in Section 4 won't fail on a missing value.
* `year` → we will leave the two missing years missing, but store them properly as a *nullable integer* (see below).

In [11]:
beatles['lead_vocal'] = beatles['lead_vocal'].fillna('None (instrumental)')
beatles['album'] = beatles['album'].fillna('Unreleased')
beatles['album_debut'] = beatles['album_debut'].fillna('Unreleased')

for col in ['styles', 'themes', 'moods']:
    beatles[col] = beatles[col].fillna('')

beatles[descriptive_cols].isna().sum()

year           2
album          0
album_debut    0
songwriter     0
lead_vocal     0
genre          0
styles         0
themes         0
moods          0
dtype: int64

### Missing as information: a True/False "charted" column

Instead of filling the chart columns, we create **Boolean** (True/False) columns that record whether a value exists. `.notna()` returns `True` where there *is* a value.

The `cover` column works the same way: it holds `'Y'` for covers and `NaN` for Beatles originals, so it is really a True/False column in disguise.

In [12]:
beatles['billboard_top50'] = beatles['top_50_billboard'].notna()
beatles['is_cover'] = beatles['cover'] == 'Y'
beatles = beatles.drop(columns=['cover'])


# breakdown of all the combinations of songs that are covers and did not chart
beatles[['billboard_top50', 'is_cover']].value_counts()

billboard_top50  is_cover
False            False       168
                 True         61
True             False        44
                 True          5
Name: count, dtype: int64

## 4. Wrong or Inconsistent Values: `replace()`, `map()`, and `apply()`

Once missing data is handled, the next job is fixing values that are **wrong, inconsistent, or in an awkward format**. Pandas gives us three main tools. Each works on a whole column at once, but they differ in how much *logic* they can handle:

| Method | What it does | Best for | Our example |
|---|---|---|---|
| `.replace(dictionary)` | swaps **specific** values; everything else stays the same | a handful of known fixes | album names |
| `.map(dictionary)` | translates **every** value by looking it up; values *not* in the dictionary become `NaN` | a complete translation from codes to labels | musical keys and modes |
| `.apply(function)` | runs **your own function** on every value | anything that needs logic (splitting, checking, rebuilding) | turning label strings into **lists** |

A good rule of thumb: start with the simplest tool that does the job. Use `replace` for a few fixes, `map` for a full lookup, and `apply` when you need to write steps of your own.

### 4a. `replace()`: fixing album names

A quick way to spot inconsistencies is to look at the **unique values** in a column, with their counts:

In [13]:
beatles['album'].value_counts()

album
The Beatles                                31
Live at the BBC                            27
Past Masters (Vols. 1 & 2 / Remastered)    26
Anthology 1                                19
Abbey Road                                 17
Revolver                                   14
Please Please Me                           14
Help!                                      14
With the Beatles                           14
Rubber Soul                                14
Beatles for Sale                           14
A Hard Day's Night                         13
Sgt. Pepper's Lonely Hearts Club Band      13
Let It Be                                  12
Magical Mystery Tour                       11
Anthology 3                                 9
Anthology 2                                 4
Yellow Submarine                            4
Live At The BBC                             3
On Air – Live at the BBC Volume 2           1
Abbey Road (Super Deluxe Edition)           1
On Air - Live At The BBC (Vo

Notice the problems:

* `Live at the BBC` and `Live At The BBC` (capitalization)
* `On Air – Live at the BBC Volume 2` and `On Air - Live At The BBC (Vol.2)` (different dashes and wording)
* `Abbey Road (Super Deluxe Edition)` is a reissue of `Abbey Road`

We need to change just three values and leave all the others alone: a job for `replace()`. The dictionary's *keys* are the old values; its *values* are the new ones.

> A handy trick for building such a dictionary: `dict.fromkeys(beatles['album'].unique())` makes a dictionary with every unique value as a key, ready for you to fill in.

NB:  the `replace()` only touches the values we specify in the dictionary.  This is good if we want to leave certain values alone.  But bad if we want to make sure we correct every possible value.  We could have missed some of them!

In [14]:
# dictionary that specifies how we want to make the replacements
album_fixes = {
    'Live At The BBC': 'Live at the BBC',
    'On Air - Live At The BBC (Vol.2)': 'On Air – Live at the BBC Volume 2',
    'Abbey Road (Super Deluxe Edition)': 'Abbey Road',
}
beatles['album'] = beatles['album'].replace(album_fixes)
beatles['album'].value_counts()

album
The Beatles                                31
Live at the BBC                            30
Past Masters (Vols. 1 & 2 / Remastered)    26
Anthology 1                                19
Abbey Road                                 18
With the Beatles                           14
Beatles for Sale                           14
Revolver                                   14
Rubber Soul                                14
Help!                                      14
Please Please Me                           14
Sgt. Pepper's Lonely Hearts Club Band      13
A Hard Day's Night                         13
Let It Be                                  12
Magical Mystery Tour                       11
Anthology 3                                 9
Anthology 2                                 4
Yellow Submarine                            4
On Air – Live at the BBC Volume 2           2
Rock 'n' Roll Music                         1
Unreleased                                  1
Name: count, dtype: int64

### 4b. `map()`: from key numbers to key names

Spotify stores `key` as a number (0 = C, 1 = C♯/D♭, ..., 11 = B) and `mode` as 1 (major) or 0 (minor). That's fine for a computer, but readable labels are better for grouping and charting.

Here we want to translate **every** value, so `map()` is the right choice. (If a value were missing from our dictionary, `map()` would return `NaN`, a useful warning that our lookup is incomplete.) We keep the original number columns and add new label columns alongside them.

In [15]:
key_names = {0: 'C', 1: 'C#/Db', 2: 'D', 3: 'D#/Eb', 4: 'E', 5: 'F',
             6: 'F#/Gb', 7: 'G', 8: 'G#/Ab', 9: 'A', 10: 'A#/Bb', 11: 'B'}
mode_names = {1: 'major', 0: 'minor'}

beatles['key_name'] = beatles['key'].map(key_names)
beatles['mode_name'] = beatles['mode'].map(mode_names)

beatles[['title', 'key', 'key_name', 'mode', 'mode_name']].head()

,title,key,key_name,mode,mode_name
1,12-bar Original,9.0,A,1.0,major
3,A Day in the Life,4.0,E,0.0,minor
4,A Hard Day's Night,0.0,C,1.0,major
5,A Shot of Rhythm and Blues,2.0,D,1.0,major
6,A Taste of Honey,1.0,C#/Db,0.0,minor


### 4c. `apply()`: turning label strings into lists

The four **categorical label** columns each hold *several labels in a single string*:

In [16]:
beatles[['title', 'genre', 'styles', 'themes', 'moods']].head(3)

,title,genre,styles,themes,moods
1,12-bar Original,Blues,,,Passionate
3,A Day in the Life,"Psychedelic Rock, Art Rock, Pop/Rock","British Psychedelia, Contemporary Pop/Rock, Ea...",,Ambitious Complex Dramatic Dreamy Eerie Elabor...
4,A Hard Day's Night,"Rock, Electronic, Pop/Rock","British Invasion, Contemporary Pop/Rock, Early...","Everyday Life, In Love, Relationships",Carefree Cheerful Explosive Marching Optimisti...


To Python, `'Psychedelic Rock, Art Rock, Pop/Rock'` is one long piece of text, not three genres. We can't easily count genres or ask "which songs are Art Rock?" until each label stands on its own. Notice also that `moods` separates its labels with **spaces**, while the others use **commas**.

The solution is to turn each string into a Python **list**: `['Psychedelic Rock', 'Art Rock', 'Pop/Rock']`. This takes several steps (split the text, strip extra spaces from each label, skip empty pieces), so we write a small function and hand it to `apply()`:

In [17]:
def to_label_list(text, separator=','):
    labels = []
    for label in text.split(separator):
        label = label.strip()          # remove extra spaces
        if label != '':                # skip empty pieces
            labels.append(label)
    return labels

# always test a function on an example first
to_label_list('Psychedelic Rock, Art Rock, Pop/Rock')

['Psychedelic Rock', 'Art Rock', 'Pop/Rock']

Now apply it to each column. For `moods` we pass `separator=' '`: `apply()` hands any extra arguments on to our function. (This works because multi-word moods are joined with a slash, like `Laid-Back/Mellow`.)

In [18]:
for col in ['genre', 'styles', 'themes']:
    beatles[col] = beatles[col].apply(to_label_list)

beatles['moods'] = beatles['moods'].apply(to_label_list, separator=' ')

beatles[['title', 'genre', 'styles', 'themes', 'moods']].head(3)

,title,genre,styles,themes,moods
1,12-bar Original,[Blues],[],[],[Passionate]
3,A Day in the Life,"[Psychedelic Rock, Art Rock, Pop/Rock]","[British Psychedelia, Contemporary Pop/Rock, E...",[],"[Ambitious, Complex, Dramatic, Dreamy, Eerie, ..."
4,A Hard Day's Night,"[Rock, Electronic, Pop/Rock]","[British Invasion, Contemporary Pop/Rock, Earl...","[Everyday Life, In Love, Relationships]","[Carefree, Cheerful, Explosive, Marching, Opti..."


Each cell now holds a list, and list-friendly tools work. For example, `.str.len()` counts the labels in each list:

In [19]:
beatles['genre'].str.len().value_counts().sort_index()

genre
1     54
2    141
3     61
4     17
5      4
6      1
Name: count, dtype: int64

These lists are exactly what we need for **Notebook B**, where we give each label its own row with `explode()`.

### 4d. The labels themselves may need cleaning too

Turning strings into lists is a start, but the *items* in the lists can still contain typos or inconsistent spellings. For example, look at the genres for "Helter Skelter", and at the songs whose genres include `'Hard rock'` (lowercase *r*):

In [20]:
beatles[beatles['title'] == 'Helter Skelter'][['title', 'genre']]

,title,genre
88,Helter Skelter,"[Hard Rock, Heavy Metal[, Proto-Punk, Pop/Rock]"


In [21]:
beatles[beatles['genre'].apply(lambda labels: 'Hard rock' in labels)][['title', 'genre']]

,title,genre
195,Revolution,"[Hard rock, Pop/Rock]"
196,Revolution 1,"[Hard rock, Pop/Rock]"


`'Heavy Metal['` has a stray bracket (probably left over from a Wikipedia footnote), and `'Hard rock'` won't be counted together with `'Hard Rock'`. We can combine the ideas above: a **dictionary** of fixes, used inside a function that we **apply** to every list.

> `lambda labels: 'Hard rock' in labels` is a tiny one-line function. It returns `True` if `'Hard rock'` is one of the items in the list.

In [22]:
genre_fixes = {
    'Hard rock': 'Hard Rock',
    'Heavy Metal[': 'Heavy Metal',
}

def fix_labels(labels):
    # look up each label in the dictionary; if it's not there, keep it as it is
    return [genre_fixes.get(label, label) for label in labels]

beatles['genre'] = beatles['genre'].apply(fix_labels)
beatles[beatles['title'] == 'Helter Skelter'][['title', 'genre']]

,title,genre
88,Helter Skelter,"[Hard Rock, Heavy Metal, Proto-Punk, Pop/Rock]"


There are more issues like these: `Rock & Roll` vs. `Rock and Roll`, `Pop Rock` vs. `Pop/Rock`, and typos in `styles` such as `Prog-RockHard Rock` (a missing comma). **Which ones you need to fix depends on your research question.** If you are studying genres, you'll want to clean the genre labels carefully. If you are studying moods, focus there. In Notebook B you'll see an easy way to list every label with its count, which makes such problems easy to spot.

## 5. Simplifying a Column for Research: The Principal Songwriter

`apply()` is also useful for building a *new*, simpler column from a complicated one. The `songwriter` column is very detailed: `McCartney, with Lennon`, `Lennon and McCartney`, `Blackwell, Johnson, Penniman`, ... For questions like "Do Lennon and McCartney songs differ?" we need a simpler **category**. Our function returns:

* the one Beatle named **before** any `with` (the *principal* writer): `'Lennon'`, `'McCartney'`, `'Harrison'`, or `'Starkey'`
* two names joined with `&` when two Beatles are credited equally (e.g. `Lennon and McCartney`)
* `'Whole Band'` when three or more Beatles are credited equally
* `'Cover'` when no Beatle wrote the song (or it is a traditional tune)

We'll use this `songwriter_group` column throughout the next notebooks.

In [23]:
beatle_names = ['Lennon', 'McCartney', 'Harrison', 'Starkey']

def songwriter_group(credit):
    if 'Traditional' in credit:
        return 'Cover'
    principal = credit.split('with')[0]               # ignore everyone after 'with'
    writers = [name for name in beatle_names if name in principal]
    if len(writers) == 0:
        return 'Cover'
    if len(writers) == 1:
        return writers[0]
    if len(writers) == 2:
        return ' & '.join(writers)
    return 'Whole Band'

beatles['songwriter_group'] = beatles['songwriter'].apply(songwriter_group)
beatles['songwriter_group'].value_counts()

songwriter_group
McCartney               81
Lennon                  77
Cover                   69
Harrison                27
Lennon & McCartney      17
Whole Band               3
Starkey                  2
Lennon & Harrison        1
McCartney & Harrison     1
Name: count, dtype: int64

## 6. Wrong Data Types

Pandas guesses data types when it reads a CSV, and it often guesses imperfectly. `year` was read as **text**, because two entries contain *two* years (a John Lennon demo, later completed by the other Beatles):

In [24]:
beatles[beatles['year'].str.contains('/', na=False)][['title', 'year', 'album']]

,title,year,album
70,Free As a Bird,1977/1994,Anthology 1
194,Real Love,1980/1995,Anthology 2


We keep the year the **Beatles** recorded the song: `.str.split('/').str[-1]` takes the last part. Then `pd.to_numeric()` converts the text to numbers, so we can sort, compare (`year > 1965`), and plot.

We also convert columns like `key`, `mode`, and `popularity` from floats (`43.0`) to whole numbers. Note the capital **`'Int64'`**: this is Pandas' *nullable integer* type, which allows whole numbers **and** missing values (`<NA>`). The ordinary `'int64'` would fail on our two missing years.

In [25]:
beatles['year'] = beatles['year'].str.split('/').str[-1]
beatles['year'] = pd.to_numeric(beatles['year']).astype('Int64')

int_cols = ['popularity', 'duration', 'key', 'mode', 'time_signature', 'other_releases', 'covered_by']
for col in int_cols:
    beatles[col] = beatles[col].astype('Int64')

beatles[['year'] + int_cols].dtypes

year              Int64
popularity        Int64
duration          Int64
key               Int64
mode              Int64
time_signature    Int64
other_releases    Int64
covered_by        Int64
dtype: object

## 7. Duplicates

Duplicate rows inflate counts and distort averages. `.duplicated()` marks rows that repeat an earlier row.

One catch: `.duplicated()` can't compare **lists**, so we can no longer check the whole row at once. Instead we check the columns that should identify each song uniquely: the Spotify `uri` and the `title`.

In [26]:
print('duplicate URIs:  ', beatles.duplicated(subset=['uri']).sum())
print('duplicate titles:', beatles.duplicated(subset=['title']).sum())

duplicate URIs:   0
duplicate titles: 0


No duplicates. If there were, `beatles.drop_duplicates(subset=['title'])` would remove them (keeping the first copy).

## 8. Save the Clean Data as a Pickle

First, let's put the columns in a logical order: identifiers, Beatles metadata, categorical labels, Spotify audio features, then chart data. We also reset the index, since we dropped rows earlier.

In [27]:
id_cols       = ['title', 'year', 'album', 'album_debut', 'songwriter', 'songwriter_group',
                 'lead_vocal', 'is_cover']
label_cols    = ['genre', 'styles', 'themes', 'moods']
spotify_cols  = ['popularity', 'duration', 'key', 'key_name', 'mode', 'mode_name', 'tempo',
                 'time_signature', 'valence', 'danceability', 'energy', 'loudness',
                 'acousticness', 'instrumentalness', 'liveness', 'speechiness']
release_cols  = ['other_releases', 'single_a_side', 'single_b_side', 'single_certification', 'covered_by']
chart_cols    = [col for col in beatles.columns if col.startswith(('chart_', 'highest_', 'weeks_', 'top_50'))]
chart_cols    = ['billboard_top50'] + chart_cols

beatles_clean = beatles[id_cols + label_cols + spotify_cols + release_cols + chart_cols + ['uri']]
beatles_clean = beatles_clean.reset_index(drop=True)

# confirm we kept every column exactly once
print(len(beatles_clean.columns), 'columns (was', len(beatles.columns), ')')
beatles_clean.head()

49 columns (was 49 )


,title,year,album,album_debut,songwriter,songwriter_group,lead_vocal,is_cover,genre,styles,themes,moods,popularity,duration,key,key_name,mode,mode_name,tempo,time_signature,valence,danceability,energy,loudness,acousticness,instrumentalness,liveness,speechiness,other_releases,single_a_side,single_b_side,single_certification,covered_by,billboard_top50,chart_position_uk_wikipedia,chart_position_us_wikipedia,highest_position_the_guardian,weeks_on_chart_in_uk_the_guardian,weeks_at_no1_in_uk_the_guardian,highest_position_billboard,weeks_at_no1_billboard,top_50_billboard,top_50_ultimate_classic_rock,top_50_rolling_stone,top_50_nme,top_50_top50songsorg,top_50_usa_today_2017,top_50_vulture_by_bill_wyman,uri
0,12-bar Original,1965,Anthology 2,Anthology 2,"Lennon, McCartney, Harrison and Starkey",Whole Band,None (instrumental),False,[Blues],[],[],[Passionate],31,175,9,A,1,major,122.678,4,0.566,0.600,0.535,-8.178,0.00154,0.102000,0.1230,0.0298,<NA>,NaN,NaN,NaN,<NA>,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:2HvTGx5fzFGpHSyRNvXd9T
1,A Day in the Life,1967,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band,Lennon and McCartney,Lennon & McCartney,Lennon and McCartney,False,"[Psychedelic Rock, Art Rock, Pop/Rock]","[British Psychedelia, Contemporary Pop/Rock, E...",[],"[Ambitious, Complex, Dramatic, Dreamy, Eerie, ...",65,335,4,E,0,minor,163.219,4,0.175,0.364,0.457,-14.162,0.29000,0.000106,0.9220,0.0675,12,Sgt. Pepper's Lonely Hearts Club Band / With a...,A Day in the Life,NaN,27,False,63.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2.0,18.0,1.0,1.0,spotify:track:0hKRSZhUGEhKU6aNSPBACZ
2,A Hard Day's Night,1964,A Hard Day's Night,UK: A Hard Day's Night US: 1962-1966,Lennon,Lennon,"Lennon, with McCartney",False,"[Rock, Electronic, Pop/Rock]","[British Invasion, Contemporary Pop/Rock, Earl...","[Everyday Life, In Love, Relationships]","[Carefree, Cheerful, Explosive, Marching, Opti...",71,152,0,C,1,major,138.514,4,0.797,0.590,0.805,-6.481,0.13700,0.000000,0.0996,0.0371,35,A Hard Day's Night; A Hard Day's Night,Things We Said Today; I Should Have Known Better,RIAA Gold,35,True,1.0,1.0,1.0,13.0,3.0,1.0,2.0,8.0,18.0,11.0,19.0,19.0,15.0,41.0,spotify:track:5J2CHimS7dWYMImCHkEFaJ
3,A Shot of Rhythm and Blues,1963,Live at the BBC,Live at the BBC,Thompson,Cover,Lennon,True,"[R&B, Pop/Rock]","[British Invasion, Merseybeat, Contemporary Po...","[Celebration, Hanging Out, Partying]","[Amiable/Good-Natured, Innocent, Swaggering, B...",35,136,2,D,1,major,128.919,4,0.968,0.699,0.752,-9.565,0.87900,0.000237,0.2650,0.0354,<NA>,NaN,NaN,NaN,<NA>,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:5RuE7nwp4TWsRvLXG7Yjy1
4,A Taste of Honey,1963,Please Please Me,UK: Please Please Me US: The Early Beatles,"Scott, Marlow",Cover,McCartney,True,"[Pop/Rock, Jazz, Stage&Screen]","[British Invasion, Early Pop/Rock, Merseybeat,...",[Affection/Fondness],"[Earnest, Precious, Refined, Sentimental, Swee...",51,163,1,C#/Db,0,minor,101.408,3,0.412,0.420,0.372,-11.416,0.69800,0.000000,0.1040,0.0327,29,NaN,NaN,NaN,<NA>,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:7fh53ta3vAOGJMQ4i5tCHe


### Why Pickle?

A CSV stores everything as text, so when you reload it Pandas must guess the data types all over again: our `Int64` years, our True/False columns, and our **lists** of labels would all be lost. A **Pickle** (`.pkl`) saves the DataFrame *exactly* as it is in memory.

> Pickles are for your own work. Never open a pickle from a source you don't trust, and use CSV when sharing data with people who don't use Python.

In [28]:
beatles_clean.to_pickle('beatles_clean.pkl')

# check that it reloads correctly
pd.read_pickle('beatles_clean.pkl').shape

(278, 49)

### Summary

| Step | Tool |
|---|---|
| Tidy column names | a function + list comprehension |
| Find missing data | `.isna().sum()` |
| Drop unusable rows | `.dropna(subset=[...])` |
| Fill missing values | `.fillna(value)`, one column at a time |
| Missing as information | `.notna()` → True/False column |
| Fix a few specific values | `.replace(dictionary)` |
| Translate every value | `.map(dictionary)` |
| Fixes that need logic | `.apply(function)` |
| Fix data types | `pd.to_numeric()`, `.astype('Int64')` |
| Check duplicates | `.duplicated(subset=[...])` |
| Save | `.to_pickle()` |

> **Reproducibility:** it can be tempting to fix errors by hand in the spreadsheet. But if every change is made in code, as here, anyone can start from the original CSV and get exactly your result, and see every decision you made along the way.

**Next:** in **Notebook B** we load `beatles_clean.pkl` and make the data **tidy**.

**Beatles notebooks:** **A. Clean Data** · [B. Tidy Data](B_Beatles_Tidy_Data.ipynb) · [C. Find, Filter, Group](C_Beatles_Find_Filter_Group.ipynb) · [D. Charts and Graphs](D_Beatles_Charts_Graphs.ipynb)